This code detects the location of starting point of an office;

Input: List of offices + Corresponding offices 

=> Output: List of offices with coordinate information

In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import json

In [2]:
def Detect_Office(Json,Office):

    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if (Office[0] == d['inferText'][0]) or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

def Detect_Office2(Json,Office):
    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if ( d['inferText'][0] == '〇') or( d['inferText'][0] == '○') or ( d['inferText'][0] == '0')or ( d['inferText'][0] == 'O') or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(file_data):
    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

def Clova2(file_data):
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\Temp.jpg"
    with open(temp_path, "wb") as path:
        path.write(file_data)
    
    stream = open(temp_path, "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

    HH,WW=img.shape[:2]
    cropped=img[0:200,0:WW]
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
    cv2.imwrite(temp_path+"Temp.jpg",cropped)
    
    with open(temp_path+"Temp.jpg",'rb') as f:
        image = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(image).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

In [3]:
def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation)
                      for im in im_list]
    return cv2.hconcat(im_list_resize)

def Check(df):
    for n in range(1,len(df)):
        try:
            
            #Extract key info of office
            Row  = df.iloc[n]

            ###Collect Location information###
            Dept=Row["Dept"]
            Office=Row["Office"]

            print('['+str(n)+',"'+Dept+'","'+Office+'"]')
            StrPage=dta[Year][Dept][Office]['StrPage']
            StrPart=dta[Year][Dept][Office]['StrPart']
            StrLoc=int(dta[Year][Dept][Office]['StrLocation'])

            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+StrPart+".jpg",'rb')
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            oldimg=img.copy()[0:200,0:WW]

            EndLoc=int(dta[Year][Dept][Office]['EndLocation'])
            EndPage=int(dta[Year][Dept][Office]['EndPage'])
            EndPart=dta[Year][Dept][Office]['EndPart']


            #Start#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+StrPart+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(StrLoc,0),(StrLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            #End#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(EndPage)+"\\"+EndPart+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(EndLoc,0),(EndLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            cv2.imshow(Dept+Office,oldimg)
            cv2.waitKey(0)
        except:
            print('Failed at '+Dept+Office)
    cv2.destroyAllWindows()
    cv2.waitKey(0)

In [4]:
def Show(n,Part,StrLoc):
    Row=df.iloc[n]
    Page=Row['Page']
    
    path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+Part+".jpg",'rb')
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    
    HH,WW=img.shape[:2]
    cv2.line(img,(StrLoc,0),(StrLoc,HH),(225,0,0),2)
    
    cv2.imshow('Projection',img)
    cv2.waitKey(0)

#Step 1

Fill in years to refer.
Remember to keep it as a string. NOT as float.

In [17]:
Year='1940'
Showa='15'
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
os.chdir(path)
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

In [ ]:
file_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+Year+'\\DataFrame.json'
with open(file_path, encoding="utf-8") as f:
    dta = json.loads(f.read())

#Step 2

The following code will extract the location of offices.

In [62]:
n=0
FailedList=[]
#Extract key info of office
Row  = df.iloc[n]

Page=int(Row["Page"])
Office=Row["Office"]
Dept=Row['Dept']

###Insert Starting page information to motherframe###
dta[Year][Dept][Office]={}
dta[Year][Dept][Office]["Starting_Page"]=Page
print(dta[Year][Dept][Office])

###Collect Location information###
#Part1
with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
    file_data = f.read()
#Convert to json via CLOVA
Json=Clova(file_data)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
if XCoord_Unit==None:
    #Add to motherframe
    dta[str(Year)][Dept][Office]["StrLocation"]='NA'
else:
    dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
    dta[str(Year)][Dept][Office]["StrPart"]='Top'
    print(Office+ 'success: at row'+str(n))

#Part2
with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
    file_data = f.read()
#Convert to json via CLOVA
Json=Clova(file_data)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
if XCoord_Unit==None:
    #Add to motherframe
    dta[str(Year)][Dept][Office]["StrLocation"]='NA'
    print(Office+ 'failed')
    FailedList.append(list((n,Dept,Office)))
else:
    dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
    dta[str(Year)][Dept][Office]["StrPart"]='Btm'
    print(Office+ 'success: at row'+str(n))

{'Starting_Page': 3}
秘書課success: at row0
秘書課success: at row0


In [63]:
#Test code| Version 2#
#Show Working office#
for n in range(1,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']
    print('['+str(n)+',"'+Dept+'","'+Office+'"]')
    
    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Insert Starting page information to motherframe###
    dta[Year][Dept][Office]={}
    dta[Year][Dept][Office]["StrPage"]=Page
    print(dta[Year][Dept][Office])

    ###Collect Location information###
    #Part1
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
        file_data = f.read()
    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Top Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))
    
    if XCoord_Unit!=None:
        continue
        
    #Part2
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
        file_data = f.read()

    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Bottom Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))

[1,"総務局","文書課"]
{'StrPage': 3}
Top LocNone
Bottom Loc398
文書課success: at row1
[2,"総務局","人事課"]
{'StrPage': 5}
Top Loc459
人事課success: at row2
[3,"総務局","吏務課"]
{'StrPage': 5}
Top Loc107
吏務課success: at row3
[4,"総務局","議案課"]
{'StrPage': 5}
Top LocNone
Bottom Loc108
議案課success: at row4
[5,"総務局","企画課"]
{'StrPage': 6}
Top LocNone
Bottom LocNone
企画課failed
[6,"総務局","都市計画課"]
{'StrPage': 6}
Top LocNone
Bottom LocNone
都市計画課failed
[7,"総務局","統計課"]
{'StrPage': 8}
Top Loc250
統計課success: at row7
[8,"総務局","情報課"]
{'StrPage': 10}
Top Loc383
情報課success: at row8
[9,"監査部","市務監察課"]
{'StrPage': 10}
Top LocNone
Bottom LocNone
市務監察課failed
[10,"監査部","区務監察課"]
{'StrPage': 11}
Top LocNone
Bottom LocNone
区務監察課failed
[11,"財務局","会計課"]
{'StrPage': 11}
Top LocNone
Bottom Loc245
会計課success: at row11
[12,"財務局","主計課"]
{'StrPage': 12}
主計課failed
[13,"財務局","主税課"]
{'StrPage': 13}
Top Loc257
主税課success: at row13
[14,"財務局","公債課"]
{'StrPage': 16}
公債課failed
[15,"財務局","用品課"]
{'StrPage': 16}
Top Loc48
用品課success: at row15
[16,"財務局","購買課"

In [64]:
FailedList2=[]
for Office in FailedList:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Collect Location information###
    #Part1
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
        file_data = f.read()
    #Convert to json via CLOVA
    try:
        Json=Clova2(file_data)
    except:
        print(Office+ 'failed')
        FailedList2.append(list((n,Dept,Office)))
        continue
    
    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office2(Json,Office)
    print('Top Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))
    
    if XCoord_Unit!=None:
        continue
        
    #Part2
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
        file_data = f.read()

    #Convert to json via CLOVA
    try:
        Json=Clova2(file_data)
    except:
        print(Office+ 'failed')
        FailedList2.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office2(Json,Office)
    print('Bottom Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList2.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))

Top Loc189
企画課success: at row5
Top LocNone
Bottom Loc57
都市計画課success: at row6
Top Loc381
市務監察課success: at row9
Top Loc418
区務監察課success: at row10
Top Loc445
学校営業課success: at row20
Top Loc443
町会課success: at row23
Top LocNone
Bottom Loc269
防衛課success: at row26
Top Loc153
計画課success: at row30
Top Loc153
事業課success: at row31
Top LocNone
Bottom Loc342
庶務課success: at row32
Top Loc410
学校職員課success: at row33
Top Loc410
視学課success: at row35
Top Loc372
医務課success: at row52
Top Loc372
分院及学校success: at row54
Top Loc396
治水工事課success: at row73
Top LocNone
Bottom Loc346
区土木課役務者success: at row75
Top LocNone
Bottom LocNone
経理課failed
Top Loc272
臨時電源調査課success: at row104
Top Loc318
病院success: at row105


In [65]:
for Office in FailedList2:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    dta[str(Year)][Dept][Office]["StrLocation"]=0
    dta[str(Year)][Dept][Office]["StrPart"]='Top'
    dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
    dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
    dta[str(Year)][ExDept][ExOffice]["EndLocation"]=0
    dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       

#Step 3

Check for errors.
Manually type in the index, department name, and office name of erroneous department-office pair to a seperate list.

In [66]:
Check(df)

[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"総務局","情報課"]
[9,"監査部","市務監察課"]
[10,"監査部","区務監察課"]
[11,"財務局","会計課"]
Failed at 財務局会計課
[12,"財務局","主計課"]
Failed at 財務局主計課
[13,"財務局","主税課"]
Failed at 財務局主税課
[14,"財務局","公債課"]
Failed at 財務局公債課
[15,"財務局","用品課"]
[16,"財務局","購買課"]
[17,"財務局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"市民局","庶務課"]
[22,"市民局","区政課"]
[23,"市民局","町会課"]
[24,"市民局","體力課"]
[25,"市民局","公園課"]
[26,"市民局","防衛課"]
[27,"市民局","観光課"]
Failed at 市民局観光課
[28,"国民精神総動員部","第一課"]
Failed at 国民精神総動員部第一課
[29,"国民精神総動員部","第二課"]
[30,"記念事業部","計画課"]
[31,"記念事業部","事業課"]
[32,"教育局","庶務課"]
[33,"教育局","学校職員課"]
[34,"教育局","学務課"]
[35,"教育局","視学課"]
[36,"教育局","青年教育課"]
[37,"教育局","社会教育課"]
Failed at 教育局社会教育課
[38,"教育局","学校体育課"]
Failed at 教育局学校体育課
[39,"教育局","教育研究所"]
[40,"厚生局","庶務課"]
[41,"厚生局","軍事援護課"]
[42,"厚生局","児童課"]
[43,"厚生局","保護課"]
[44,"厚生局","福利課"]
[45,"厚生局","衛生課"]
[46,"厚生局","防疫課"]
[47,"清掃部","監理課"]
[48,"清掃部","作業課"]
[49,"清掃部","計書課

Type in the department-offices with errors.

In [23]:
Type1=[[1,"総務局","文書課"],
       [3,"総務局","吏務課"],
       [5,"総務局","企画課"],
       [6,"総務局","都市計画課"],
       [9,"監査部","市務監察課"],
       [10,"監査部","区務監察課"],
       [23,"市民局","町会課"],
       [29,"国民精神総動員部","第二課"],
       [30,"記念事業部","計画課"],
       [33,"教育局","学校職員課"],
       [35,"教育局","視学課"],
       [36,"教育局","青年教育課"],
       [37,"教育局","社会教育課"],
       [38,"教育局","学校体育課"],
       [39,"教育局","教育研究所"],
       [41,"厚生局","軍事援護課"],
       [45,"厚生局","衛生課"],
       [49,"清掃部","計書課"],
       [54,"養育院","分院及学校"],
       [58,"経済局","商工課"],
       [65,"中央卸売市場","管理課"],
       [67,"中央卸売市場","副収入役室"],
       [73,"土木局","治水工事課"],
       [74,"土木局","土木試験所"],
       [79,"港湾局","工事課"],
       [80,"港湾局","港務序"],
       [81,"港湾局","臨時飛行場建設事務所"],
       [85,"水道局","給水課"],
       [93,"電気局","経理課"],
       [95,"電気局","交通調整課"],
       [97,"運輸部","運輸課"],
       [99,"運輸部","車両課"],
       [103,"電燈部","電力課"],
       [104,"電燈部","臨時電源調査課"],
       [107,"電気研究所","試験課"]]+FailedList2

#Step 4

See how much errors were observed in the first trial.

In [24]:
FailRate=len(Type1)/len(df)
if FailRate<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-FailRate))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate
display(DF)
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index=False)

残念...Success Rate is 0.6146788990825688


,Year,Parts,WritePage,Page,PageFin,Office,OfficeFin,Unit,UnitFin,Position,PositionFin,Data
0,1934,1,1,0.985714286,.,0.9,.,NaN,.,0.08,.,.
1,1935,1,1,1,.,0.984375,.,NaN,.,-0.09375,.,.
2,1936,1,0.235294118,.,.,0.235294118,.,NaN,.,.,.,.
3,1937,1,1,1,.,0.865671642,.,NaN,.,.,.,.
4,1938,1,1,1,.,1,.,0.988095238,.,0.98816568,.,0.619957537
5,1939,1,1,1,.,0.927835052,.,1,.,.,.,.
6,1940,2,1,1,.,0.614679,.,NaN,.,.,.,.
7,1941,2,1,1,.,0.565217391,Finished!!!,NaN,.,.,.,.
8,1942,2,.,1.0,.,.,.,NaN,.,.,.,.
9,1943,2,1,.,.,.,.,.,.,.,.,.


In [67]:
k=0
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=190
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

総務局 文書課


In [68]:
k=1
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=80
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

総務局 吏務課


In [69]:
k=2
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=190
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

総務局 企画課


In [70]:
k=3
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=60
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

総務局 都市計画課


In [71]:
k=4
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=420
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

監査部 市務監察課


In [72]:
k=5
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=420
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

監査部 区務監察課


In [73]:
k=6
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=540
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

市民局 町会課


In [77]:
k=7
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=160
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

国民精神総動員部 第二課


In [83]:
k=8
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=140
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

記念事業部 計画課


In [86]:
k=9
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=420
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 学校職員課


In [88]:
k=10
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=80
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 視学課


In [97]:
k=11
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=480
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 青年教育課


In [102]:
k=12
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=270
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 社会教育課


30

In [106]:
k=13
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=290
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 学校体育課


29

In [109]:
k=14
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=540
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 教育研究所


28

In [113]:
k=15
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=175
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

厚生局 軍事援護課


27

In [118]:
k=16
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=460
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

厚生局 衛生課


26

In [123]:
k=17
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=460
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

清掃部 計書課


25

In [128]:
k=18
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=80
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

養育院 分院及学校


24

In [131]:
k=19
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=360
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

経済局 商工課


23

In [135]:
k=20
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=370
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

中央卸売市場 管理課


22

In [140]:
k=21
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=420
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

中央卸売市場 副収入役室


21

In [141]:
k=22
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=180
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 治水工事課


20

In [145]:
k=23
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=350
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 土木試験所


19

In [148]:
k=24
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=370
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

港湾局 工事課


18

In [151]:
k=25
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=170
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

港湾局 港務序


17

In [154]:
k=26
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=170
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

港湾局 臨時飛行場建設事務所


16

In [158]:
k=27
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=190
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

水道局 給水課


15

In [162]:
k=28
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=190
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電気局 経理課


14

In [166]:
k=29
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=230
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電気局 交通調整課


13

In [170]:
k=30
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=260
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

運輸部 運輸課


12

In [178]:
k=31
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=280
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

運輸部 車両課


11

In [182]:
k=32
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=250
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電燈部 電力課


10

In [187]:
k=33
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=280
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電燈部 臨時電源調査課


9

In [192]:
k=34
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=185
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電気研究所 試験課


8

In [195]:
k=35
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=190
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

総務局 企画課


7

In [199]:
k=36
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=60
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

総務局 都市計画課


6

In [202]:
k=37
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=420
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

監査部 区務監察課


5

In [204]:
k=38
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=420
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 学校職員課


4

In [206]:
k=39
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=180
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 治水工事課


3

In [211]:
k=40
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=195
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電気局 経理課


2

In [216]:
k=41
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=280
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電燈部 臨時電源調査課


1

In [36]:
for n in range(0,len(df)):
    try:
        Row=df.iloc[n]
        ExRow=df.iloc[n-1]

        Office,Dept=Row['Office'],Row['Dept']
        ExOffice,ExDept=ExRow['Office'],ExRow['Dept']    

        StrPage=Row['Page']
        ExStrPage=ExRow['Page']

        StrLoc=dta[Year][Dept][Office]['StrLocation']
        StrPart=dta[Year][Dept][Office]['StrPart']

        dta[Year][ExDept][ExOffice]['EndPage']=StrPage
        dta[Year][ExDept][ExOffice]['EndLocation']=StrLoc
        dta[Year][ExDept][ExOffice]['EndPart']=StrPart
        
        dta[Year][ExDept][ExOffice]['Page_Range']=list(range(ExStrPage,StrPage+1))

    except:
        continue

In [24]:
Check(df)

[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"総務局","情報課"]
[9,"監査部","市務監察課"]
[10,"監査部","区務監察課"]
[11,"財務局","会計課"]
[12,"財務局","主計課"]
[13,"財務局","主税課"]
Failed at 財務局主税課
[14,"財務局","公債課"]
Failed at 財務局公債課
[15,"財務局","用品課"]
[16,"財務局","購買課"]
[17,"財務局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"市民局","庶務課"]
[22,"市民局","区政課"]
[23,"市民局","町会課"]
[24,"市民局","體力課"]
[25,"市民局","公園課"]
[26,"市民局","防衛課"]
[27,"市民局","観光課"]
[28,"国民精神総動員部","第一課"]
[29,"国民精神総動員部","第二課"]
[30,"記念事業部","計画課"]
[31,"記念事業部","事業課"]
[32,"教育局","庶務課"]
[33,"教育局","学校職員課"]
[34,"教育局","学務課"]
[35,"教育局","視学課"]
[36,"教育局","青年教育課"]
[37,"教育局","社会教育課"]
[38,"教育局","学校体育課"]
[39,"教育局","教育研究所"]
[40,"厚生局","庶務課"]
[41,"厚生局","軍事援護課"]
[42,"厚生局","児童課"]
[43,"厚生局","保護課"]
[44,"厚生局","福利課"]
[45,"厚生局","衛生課"]
[46,"厚生局","防疫課"]
[47,"清掃部","監理課"]
[48,"清掃部","作業課"]
[49,"清掃部","計書課"]
[50,"養育院","庶務課"]
[51,"養育院","監護課"]
[52,"養育院","医務課"]
[53,"養育院","会計課"]
[54,"養育院","分院及学校"]
[55,"療養所","管理課"]
[56,

In [29]:
Error=[[11,"財務局","会計課"],
       [12,"財務局","主計課"],
       [13,"財務局","主税課"],
       [27,"市民局","観光課"],
       [28,"国民精神総動員部","第一課"],
       [37,"教育局","社会教育課"],
       [108,"電気研究所","研究課"],
       [14,"財務局","公債課"]]

In [18]:
k=0
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExRow=df.iloc[n-1]
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=250
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

財務局 会計課


In [233]:
k=1
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=290
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

財務局 主計課


In [265]:
k=2
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=260
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

財務局 主税課


In [240]:
k=3
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=330
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

市民局 観光課


In [246]:
k=4
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=385
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

国民精神総動員部 第一課


In [252]:
k=5
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=270
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 社会教育課


In [258]:
k=6
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=370
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

電気研究所 研究課


In [34]:
k=7
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=425
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

財務局 公債課


In [27]:
dta[Year]["財務局"]["公債課"]

{'StrPage': 16,
 'EndPage': 16,
 'EndPart': 'Top',
 'EndLocation': 48,
 'Page_Range': [16]}

In [38]:
Check(df)

[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"総務局","情報課"]
[9,"監査部","市務監察課"]
[10,"監査部","区務監察課"]
[11,"財務局","会計課"]
[12,"財務局","主計課"]
[13,"財務局","主税課"]
[14,"財務局","公債課"]
[15,"財務局","用品課"]
[16,"財務局","購買課"]
[17,"財務局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"市民局","庶務課"]
[22,"市民局","区政課"]
[23,"市民局","町会課"]
[24,"市民局","體力課"]
[25,"市民局","公園課"]
[26,"市民局","防衛課"]
[27,"市民局","観光課"]
[28,"国民精神総動員部","第一課"]
[29,"国民精神総動員部","第二課"]
[30,"記念事業部","計画課"]
[31,"記念事業部","事業課"]
[32,"教育局","庶務課"]
[33,"教育局","学校職員課"]
[34,"教育局","学務課"]
[35,"教育局","視学課"]
[36,"教育局","青年教育課"]
[37,"教育局","社会教育課"]
[38,"教育局","学校体育課"]
[39,"教育局","教育研究所"]
[40,"厚生局","庶務課"]
[41,"厚生局","軍事援護課"]
[42,"厚生局","児童課"]
[43,"厚生局","保護課"]
[44,"厚生局","福利課"]
[45,"厚生局","衛生課"]
[46,"厚生局","防疫課"]
[47,"清掃部","監理課"]
[48,"清掃部","作業課"]
[49,"清掃部","計書課"]
[50,"養育院","庶務課"]
[51,"養育院","監護課"]
[52,"養育院","医務課"]
[53,"養育院","会計課"]
[54,"養育院","分院及学校"]
[55,"療養所","管理課"]
[56,"療養所","監務課"]
[57,"経済局","庶務課"]
[58,

Fix the department-office pair with errors

First re-run the list of failed (pairs which was rejected), with a stricter OCR.

Open FailedList_2 and see if there are any other errors left.

If there are, fix it manually by guessing the starting x axis using 'Show(n,StrLoc)'

When you find the exact starting location, update the dataframe.

dta[Year][Dept][Office]={'Starting_Page': ,
                          'Office_X1': ,
                          'Ending_Page': ,
                          'Office_X2': ,
                          'Page_Range': []}

#Step 5

Do the same thing with the department-office pair which was wrongly accepted in the test.

#Step 6

Check the entire dataframe to confirm that the lines have been drawn at the correct place.

If there are errors, add the pair into the Type1 Error list and re-run step 5.

In [39]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+str(Year)+'\\'
with open(save_path+"DataFrame_Office.json", "w") as outfile:
    outfile.write(json_object)